# 🤖 NLP Assignment – 3: Chatbot using Hugging Face Transformers

**Objective:** Build a simple conversational chatbot using a pre-trained transformer model (DialoGPT) from Hugging Face that can interact with users and generate meaningful responses.

---

**Pipeline Overview:**
```
User Input → Model Processing → Response Generation → Display Output → Loop Until Exit
```

**Model Used:** `microsoft/DialoGPT-medium` — a conversational GPT-2 variant fine-tuned on Reddit dialogues.


## Step 1: Install Required Libraries

We install the Hugging Face `transformers` library and `torch` (PyTorch backend) needed for model inference.

In [ ]:
# Install Hugging Face Transformers and PyTorch (if not already installed)
!pip install transformers torch --quiet

## Step 2: Import Libraries

We import the necessary modules:
- `AutoTokenizer` – converts text to token IDs the model understands
- `AutoModelForCausalLM` – loads the pre-trained causal language model
- `torch` – PyTorch for tensor operations

In [ ]:
# Core libraries for transformer model and tokenization
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("✅ Libraries imported successfully!")

## Step 3: Load Pre-trained Model and Tokenizer

We use **DialoGPT-medium** from Microsoft — a dialogue-optimized transformer model.

- The **tokenizer** converts raw text ↔ token IDs.
- The **model** takes token IDs and generates a response.

> First run will download the model (~350 MB). Subsequent runs load from cache.

In [ ]:
# Define the model name from Hugging Face Model Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"⏳ Loading model: {MODEL_NAME} ...")

# Load the tokenizer — handles text-to-token conversion
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the causal language model — generates text token by token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode (disables dropout layers used only during training)
model.eval()

print(f"✅ Model '{MODEL_NAME}' loaded successfully!")
print(f"🔢 Model parameters: {model.num_parameters():,}")

## Step 4: Define Response Generation Function

This function handles the core chatbot logic:
1. Encodes the user's message as tokens and appends the EOS token (end-of-sequence separator)
2. Appends to the conversation history (for context-aware replies)
3. Feeds the full history to the model
4. Decodes only the **new** response tokens (skipping the input history)

**Generation parameters explained:**
| Parameter | Purpose |
|---|---|
| `max_length` | Caps total token length of input + output |
| `do_sample` | Enables sampling (creative, non-deterministic output) |
| `top_k` | Considers only top-K most likely next tokens |
| `top_p` | Nucleus sampling — picks from tokens covering top P% probability mass |
| `temperature` | Controls randomness; lower = more focused, higher = more creative |
| `pad_token_id` | Silences warnings about padding in open-ended generation |

In [ ]:
def generate_response(user_input, chat_history_ids=None):
    """
    Generates a chatbot response using DialoGPT.

    Args:
        user_input (str): The latest message from the user.
        chat_history_ids (torch.Tensor or None): Tokenized history of the conversation.

    Returns:
        tuple: (response_text, updated_chat_history_ids)
    """
    # Step 4a: Tokenize the new user input and add EOS token as turn separator
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"  # Return as PyTorch tensor
    )

    # Step 4b: Append new input to conversation history
    # On the first turn, chat_history_ids is None, so we just use the new input
    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    # Step 4c: Generate model response
    # We use no_grad() to skip gradient computation (saves memory during inference)
    with torch.no_grad():
        output_ids = model.generate(
            bot_input_ids,
            max_length=1000,          # Max total tokens (input + output)
            do_sample=True,           # Enable probabilistic sampling
            top_k=50,                 # Sample from top 50 candidate tokens
            top_p=0.95,               # Nucleus sampling threshold
            temperature=0.75,         # Moderate creativity (0=deterministic, 1=very random)
            pad_token_id=tokenizer.eos_token_id  # Suppress padding warnings
        )

    # Step 4d: Decode only the newly generated tokens (skip the input history)
    response_text = tokenizer.decode(
        output_ids[:, bot_input_ids.shape[-1]:][0],  # Slice off input tokens
        skip_special_tokens=True                      # Remove EOS and other special tokens
    )

    # Return the response and updated conversation history for the next turn
    return response_text, output_ids


print("✅ Response generation function defined!")

## Step 5: Run the Chatbot — Single Test Interaction

Before running the full loop, let's verify the model works by testing a single exchange.

In [ ]:
# Quick sanity check: test one input-response cycle
test_input = "Hello! What can you do?"

print(f"User  : {test_input}")
response, history = generate_response(test_input)
print(f"Chatbot: {response}")

## Step 6: Full Interactive Chatbot Loop

This is the main chatbot loop:
- Prints a greeting to the user
- Continuously accepts user input
- Exits cleanly if the user types `exit` or `quit`
- Maintains conversation history so the model has context across multiple turns

> **Note:** When running in Jupyter Notebook, the `input()` cell will show a text box. Type your message and press Enter to get a response.

In [ ]:
def run_chatbot():
    """
    Runs the interactive chatbot loop.
    Maintains multi-turn conversation history for context-aware responses.
    Exits when the user types 'exit' or 'quit'.
    """
    print("=" * 60)
    print("        🤖 AI Chatbot — Powered by DialoGPT")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 60)
    print("  [Type 'exit' or 'quit' to end the conversation]")
    print("=" * 60)

    chat_history_ids = None  # Initialize empty conversation history

    while True:
        # Step 6a: Get user input from console
        user_input = input("\nYou     : ").strip()

        # Step 6b: Check for exit condition (case-insensitive)
        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot : It was great talking to you! Goodbye! 👋")
            print("=" * 60)
            break

        # Step 6c: Skip empty input gracefully
        if not user_input:
            print("Chatbot : Please type something so I can respond!")
            continue

        # Step 6d: Generate a response using the transformer model
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        # Step 6e: Display the chatbot's reply
        print(f"Chatbot : {response}")


# Entry point — run the chatbot
run_chatbot()

## Step 7: Sample Conversation Output

Below is a demonstration of the chatbot's expected interaction flow (output may vary due to sampling):

```
============================================================
        🤖 AI Chatbot — Powered by DialoGPT
============================================================
Chatbot: Hello! I am your AI assistant. How can I help you today?
------------------------------------------------------------
  [Type 'exit' or 'quit' to end the conversation]
============================================================

You     : Hello
Chatbot : Hey there! How's it going?

You     : What is Artificial Intelligence?
Chatbot : Artificial Intelligence is basically machines being designed to 
          think and learn like humans. It's used in a lot of areas like 
          self-driving cars, virtual assistants, and medical diagnosis.

You     : Who created Python?
Chatbot : Python was created by Guido van Rossum. He started working on 
          it in the late 1980s and released it in 1991.

You     : Thank you
Chatbot : No problem! Feel free to ask if you have more questions.

You     : exit
Chatbot : It was great talking to you! Goodbye! 👋
============================================================
```

## Summary

| Component | Details |
|---|---|
| **Model** | `microsoft/DialoGPT-medium` (Hugging Face) |
| **Framework** | PyTorch + Hugging Face Transformers |
| **Tokenizer** | `AutoTokenizer` (GPT-2 BPE tokenizer) |
| **Generation Strategy** | Top-K + Top-P (Nucleus) Sampling with Temperature |
| **Conversation Memory** | Full history concatenation per turn |
| **Exit Condition** | User types `exit` or `quit` |

### Key Concepts Demonstrated
- ✅ Loading a pre-trained causal language model from Hugging Face Hub
- ✅ Tokenizing and encoding text inputs for transformer models
- ✅ Multi-turn conversation with history-aware context
- ✅ Controlled text generation with sampling parameters
- ✅ Building an interactive console-based conversational loop

---
*Assignment NLP – 3 | Data Science Internship | February 2026*